# Exploratory Data Analysis — Zaragoza Solar Forecast

Before data preprocessing (outlier removal, imputation, transformations), we first look at:

- Scatter plot matrix to see pairwise feature relationships and feature-target relationships
- Histograms of missingness to check if gaps are systematic or random
- Distribution inspection (mean vs median, heavy tails, skewness)

**Contents:**
1. Load data & overview
2. Descriptive statistics & distributions
3. Missing data analysis (ESIOS gaps)
4. Scatter plot matrix
5. Correlation heatmap
6. Time series patterns (diurnal & seasonal)
7. Physics validation (clear-sky vs observed, generation vs irradiance)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=0.9)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.max_open_warning'] = 50

## 1. Load data & overview

In [ ]:
df = pd.read_csv('data/zaragoza_assembled_dataset.csv', parse_dates=['datetime_utc'], index_col='datetime_utc')

print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0]} to {df.index[-1]}')
print(f'Frequency: hourly ({len(df)} hours = {len(df)/24:.0f} days = {len(df)/8760:.1f} years)')
print(f'\nColumns ({len(df.columns)}):')
for col in df.columns:
    print(f'  {col:30s}  {df[col].dtype}')

df.head()

## 2. Descriptive statistics & distributions

We compute basic descriptive statistics as a starting point.

In [ ]:
desc = df.describe().T
desc['median'] = df.median()
desc['skew'] = df.skew()
desc['kurtosis'] = df.kurtosis()
desc['mean_vs_median'] = desc['mean'] - desc['median']

# Highlight: large mean-median gap signals skewness/outliers
desc[['count', 'mean', 'median', 'mean_vs_median', 'std', 'min', 'max', 'skew', 'kurtosis']]

### 2a. Feature distributions (histograms)

Looking at the shape of each feature's distribution, for later normalization/standardization.

In [ ]:
# Exclude cyclical calendar features from distribution plots (they're sin/cos by construction)
feature_cols = [c for c in df.columns if c not in ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']]

fig, axes = plt.subplots(4, 3, figsize=(15, 14))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
    
    # Mean and median lines
    mean_val = data.mean()
    median_val = data.median()
    ax.axvline(mean_val, color='red', linewidth=1.2, linestyle='--', label=f'mean={mean_val:.1f}')
    ax.axvline(median_val, color='orange', linewidth=1.2, linestyle='-', label=f'median={median_val:.1f}')
    
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)

# Hide unused subplot
for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature distributions (with mean vs median)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 2b. Box plots

As a complement to histograms to spot outliers and compare spread/symmetry across features.

In [ ]:
# Separate into groups with comparable scales for readability
groups = {
    'Temperature & Dewpoint (C)': ['temperature_2m_C', 'dewpoint_2m_C'],
    'Irradiance (W/m2)': ['ssrd_wm2', 'strd_wm2', 'clearsky_ghi', 'clearsky_dni', 'clearsky_dhi'],
    'Other weather': ['surface_pressure_hPa', 'total_precip_mm'],
    'Solar angles (deg)': ['solar_zenith', 'solar_azimuth'],
    'Target': ['pv_generation_gwh'],
}

fig, axes = plt.subplots(1, len(groups), figsize=(18, 5))
for ax, (title, cols) in zip(axes, groups.items()):
    df[cols].boxplot(ax=ax, vert=True)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Box plots by feature group', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Missing data analysis

In [ ]:
is_missing = df['pv_generation_gwh'].isna()
n_missing = is_missing.sum()
n_total = len(df)
print(f'Missing target values: {n_missing}/{n_total} ({n_missing/n_total*100:.1f}%)')

### 3a. When are the gaps? (temporal pattern)

In [ ]:
# Missing hours per month
missing_per_month = is_missing.groupby(df.index.to_period('M')).sum()
total_per_month = is_missing.groupby(df.index.to_period('M')).count()
pct_missing_per_month = (missing_per_month / total_per_month * 100)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

pct_missing_per_month.plot(kind='bar', ax=ax1, color='coral', edgecolor='none')
ax1.set_ylabel('% hours missing')
ax1.set_title('ESIOS target: % missing per month')
ax1.set_ylim(0, 100)
ax1.axhline(n_missing/n_total*100, color='grey', linestyle='--', linewidth=0.8, label=f'overall: {n_missing/n_total*100:.1f}%')
ax1.legend()

missing_per_month.plot(kind='bar', ax=ax2, color='coral', edgecolor='none')
ax2.set_ylabel('# hours missing')
ax2.set_title('ESIOS target: absolute missing hours per month')
ax2.set_xlabel('')

# Only show every 3rd x-tick label
for ax in [ax1, ax2]:
    labels = ax.get_xticklabels()
    for i, label in enumerate(labels):
        if i % 3 != 0:
            label.set_visible(False)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Missing by hour of day — is there a diurnal pattern?
missing_by_hour = is_missing.groupby(df.index.hour).mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))
missing_by_hour.plot(kind='bar', ax=ax, color='coral', edgecolor='none')
ax.set_ylabel('% hours missing')
ax.set_xlabel('Hour of day (UTC)')
ax.set_title('ESIOS target: % missing by hour of day')
ax.axhline(n_missing/n_total*100, color='grey', linestyle='--', linewidth=0.8, label=f'overall: {n_missing/n_total*100:.1f}%')
ax.legend()
plt.tight_layout()
plt.show()

### 3b. Missingness vs features (the lecture's histogram test)

If the distribution of a feature looks different for rows with missing target vs rows with present target, then the missingness is **non-random** with respect to that feature.

In [ ]:
check_cols = ['temperature_2m_C', 'ssrd_wm2', 'clearsky_ghi', 'surface_pressure_hPa', 'solar_zenith']

fig, axes = plt.subplots(1, len(check_cols), figsize=(18, 4))
for ax, col in zip(axes, check_cols):
    ax.hist(df.loc[~is_missing, col], bins=50, alpha=0.6, density=True, label='target present', color='steelblue')
    ax.hist(df.loc[is_missing, col], bins=50, alpha=0.6, density=True, label='target missing', color='coral')
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle('Feature distributions: target-present vs target-missing rows', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 3c. Visualize the gaps as a timeline

To see if the gaps are scattered or clustered in contiguous blocks.

In [ ]:
# Show gaps as a binary timeline (1 = missing, 0 = present)
fig, ax = plt.subplots(figsize=(16, 2.5))
ax.fill_between(df.index, 0, is_missing.astype(int), step='mid', color='coral', alpha=0.7)
ax.set_yticks([0, 1])
ax.set_yticklabels(['present', 'missing'])
ax.set_title('ESIOS target: missing data timeline')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Largest contiguous gaps
gap_runs = is_missing.astype(int).diff().fillna(0)
gap_starts = df.index[gap_runs == 1]
gap_ends = df.index[gap_runs == -1]

# Handle edge case: if the series starts or ends with a gap
if is_missing.iloc[0]:
    gap_starts = gap_starts.insert(0, df.index[0])
if is_missing.iloc[-1]:
    gap_ends = gap_ends.append(pd.DatetimeIndex([df.index[-1]]))

gaps = pd.DataFrame({'start': gap_starts[:len(gap_ends)], 'end': gap_ends[:len(gap_starts)]})
gaps['duration_hours'] = (gaps['end'] - gaps['start']).dt.total_seconds() / 3600
gaps = gaps.sort_values('duration_hours', ascending=False)

print(f'Number of gap blocks: {len(gaps)}')
print(f'\nLargest 10 gaps:')
gaps.head(10)

## 4. Scatter plot matrix

We include only the physically meaningful features (not the cyclical calendar encodings).
We use a subset of data points for readability (every 3rd row sampled from non-missing data).

In [ ]:
# Features for the scatter matrix (exclude calendar sin/cos)
scatter_cols = [
    'temperature_2m_C', 'dewpoint_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'clearsky_ghi',
    'solar_zenith', 'pv_generation_gwh'
]

# Subsample for speed (every 3rd row from non-missing data)
df_scatter = df[scatter_cols].dropna().iloc[::3]
print(f'Scatter matrix using {len(df_scatter)} samples')

g = sns.pairplot(
    df_scatter,
    corner=True,
    plot_kws={'alpha': 0.1, 's': 3, 'edgecolor': 'none'},
    diag_kws={'bins': 40, 'edgecolor': 'none'},
    height=1.8,
)
g.figure.suptitle('Scatter plot matrix (features + target)', y=1.01, fontsize=13)
plt.show()

### 4a. Zoomed-in: Target vs each feature individually

In [ ]:
target_col = 'pv_generation_gwh'
feature_vs_target = [c for c in scatter_cols if c != target_col]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

df_plot = df.dropna(subset=[target_col]).iloc[::3]

for i, col in enumerate(feature_vs_target):
    ax = axes[i]
    ax.scatter(df_plot[col], df_plot[target_col], alpha=0.3, s=8, c='steelblue', edgecolor='none')
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel(target_col, fontsize=8)
    ax.set_title(f'{col} vs target', fontsize=9)

for j in range(len(feature_vs_target), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Each feature vs PV generation target', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation heatmap

Highly correlated features signal redundancy — relevant for later feature selection.

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            annot_kws={'size': 8})
ax.set_title('Pearson correlation matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlations with target, sorted
target_corr = corr['pv_generation_gwh'].drop('pv_generation_gwh').sort_values(key=abs, ascending=False)
print('Feature correlations with PV generation (absolute, descending):')
for feat, r in target_corr.items():
    print(f'  {feat:30s}  r = {r:+.3f}')

In [ ]:
# Strongest feature-feature correlations
feature_cols_for_corr = [c for c in df.columns if c not in ['pv_generation_gwh', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']]
feature_corr = corr.loc[feature_cols_for_corr, feature_cols_for_corr]

corr_pairs = []
for i in range(len(feature_corr.columns)):
    for j in range(i+1, len(feature_corr.columns)):
        feat1 = feature_corr.columns[i]
        feat2 = feature_corr.columns[j]
        r = feature_corr.iloc[i, j]
        corr_pairs.append((feat1, feat2, r))

corr_pairs_sorted = sorted(corr_pairs, key=lambda x: abs(x[2]), reverse=True)
print('Strongest feature-feature correlations (absolute, descending):')
for feat1, feat2, r in corr_pairs_sorted[:15]:
    print(f'  {feat1:30s} \u2194 {feat2:30s}  r = {r:+.3f}')

## 6. Time series patterns

Solar generation follows strong diurnal and seasonal cycles.

### 6a. Full time series overview

In [ ]:
plot_cols = ['pv_generation_gwh', 'ssrd_wm2', 'clearsky_ghi', 'temperature_2m_C']

fig, axes = plt.subplots(len(plot_cols), 1, figsize=(16, 10), sharex=True)

for ax, col in zip(axes, plot_cols):
    daily = df[col].resample('D').mean()
    ax.plot(daily.index, daily.values, linewidth=0.5, color='steelblue')
    ax.set_ylabel(col, fontsize=9)
    ax.set_title(f'{col} \u2014 daily mean', fontsize=10)

axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
fig.suptitle('Time series overview (daily means)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 6b. Average diurnal profile by season

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df['season'] = df.index.month.map(get_season)
df['hour'] = df.index.hour

diurnal_cols = ['pv_generation_gwh', 'ssrd_wm2', 'clearsky_ghi', 'temperature_2m_C']
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
colors = {'Winter': '#2196F3', 'Spring': '#4CAF50', 'Summer': '#FF9800', 'Autumn': '#9C27B0'}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, col in zip(axes, diurnal_cols):
    for season in season_order:
        mask = df['season'] == season
        hourly_mean = df.loc[mask].groupby('hour')[col].mean()
        ax.plot(hourly_mean.index, hourly_mean.values, label=season, color=colors[season], linewidth=2)
    ax.set_xlabel('Hour (UTC)')
    ax.set_ylabel(col)
    ax.set_title(f'Diurnal profile: {col}')
    ax.legend()
    ax.set_xticks(range(0, 24, 3))

plt.suptitle('Average diurnal profiles by season', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 6c. Monthly aggregates

In [ ]:
monthly = df[['pv_generation_gwh', 'ssrd_wm2', 'clearsky_ghi', 'temperature_2m_C']].resample('ME').mean()
monthly['month_label'] = monthly.index.strftime('%Y-%m')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, ['pv_generation_gwh', 'ssrd_wm2', 'clearsky_ghi', 'temperature_2m_C']):
    ax.bar(range(len(monthly)), monthly[col], color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(f'Monthly mean: {col}', fontsize=10)
    ax.set_xticks(range(0, len(monthly), 6))
    ax.set_xticklabels(monthly['month_label'].iloc[::6], rotation=45, fontsize=8)

plt.suptitle('Monthly aggregates (2023-2025)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Physics validation

Sanity checks to confirm the data is physically consistent.

### 7a. Clear-sky GHI should upper-bound observed SSRD

In [ ]:
# Only daytime hours (solar zenith < 90)
daytime = df[df['solar_zenith'] < 90].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: observed vs clear-sky
ax1.scatter(daytime['clearsky_ghi'], daytime['ssrd_wm2'], alpha=0.05, s=2, c='steelblue', edgecolor='none')
lim = max(daytime['clearsky_ghi'].max(), daytime['ssrd_wm2'].max()) * 1.05
ax1.plot([0, lim], [0, lim], 'r--', linewidth=1, label='1:1 line')
ax1.set_xlabel('Clear-sky GHI (W/m2)')
ax1.set_ylabel('ERA5 SSRD (W/m2)')
ax1.set_title('Observed vs clear-sky irradiance (daytime only)')
ax1.legend()
ax1.set_xlim(0, lim)
ax1.set_ylim(0, lim)

# Clear-sky index (kt = ssrd / clearsky_ghi)
daytime_bright = daytime[daytime['clearsky_ghi'] > 50]
kt = daytime_bright['ssrd_wm2'] / daytime_bright['clearsky_ghi']
ax2.hist(kt, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax2.axvline(1.0, color='red', linestyle='--', linewidth=1, label='kt=1 (clear sky)')
ax2.set_xlabel('Clear-sky index (kt = SSRD / clearsky GHI)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of clear-sky index')
ax2.legend()

pct_above_1 = (kt > 1.05).sum() / len(kt) * 100
print(f'Clear-sky index > 1.05 (ssrd exceeds clearsky): {pct_above_1:.1f}% of daytime hours')
print(f'Median kt: {kt.median():.3f}, Mean kt: {kt.mean():.3f}')

plt.tight_layout()
plt.show()

### 7b. PV generation vs solar irradiance

Generation should track irradiance closely — dropping during cloudy conditions and to near-zero at night.

In [ ]:
# Pick a clear summer week and a cloudy winter week for comparison
summer_week = df.loc['2024-07-01':'2024-07-07']
winter_week = df.loc['2024-01-08':'2024-01-14']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

for ax, week, title in [(ax1, summer_week, 'Summer week (2024-07-01 to 2024-07-07)'),
                         (ax2, winter_week, 'Winter week (2024-01-08 to 2024-01-14)')]:
    ax_twin = ax.twinx()
    ax.plot(week.index, week['ssrd_wm2'], color='#FF9800', linewidth=1.2, label='SSRD (W/m2)')
    ax.plot(week.index, week['clearsky_ghi'], color='#FF9800', linewidth=0.8, linestyle='--', label='Clear-sky GHI', alpha=0.6)
    ax_twin.plot(week.index, week['pv_generation_gwh'], color='#2196F3', linewidth=1.2, label='PV generation (GWh)')
    
    ax.set_ylabel('Irradiance (W/m2)', color='#FF9800')
    ax_twin.set_ylabel('PV generation (GWh)', color='#2196F3')
    ax.set_title(title)
    
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax_twin.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

fig.suptitle('PV generation vs solar irradiance: sample weeks', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 7c. Night-time generation check

PV generation at night should be near zero.

In [ ]:
nighttime = df[df['solar_zenith'] > 95].copy()
night_gen = nighttime['pv_generation_gwh'].dropna()

print(f'Night-time hours (zenith > 95): {len(nighttime)}')
print(f'Night-time generation stats:')
print(night_gen.describe())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(night_gen, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
ax.set_xlabel('PV generation (GWh)')
ax.set_ylabel('Count')
ax.set_title('Night-time PV generation distribution (solar zenith > 95 deg)')
plt.tight_layout()
plt.show()

### 7d. Temperature effect

PV efficiency decreases with higher temperatures.

In [ ]:
# Only bright daytime hours with meaningful irradiance and generation
bright = df[(df['clearsky_ghi'] > 100) & (df['ssrd_wm2'] > 100) & (df['pv_generation_gwh'].notna())].copy()
bright['gen_per_irradiance'] = bright['pv_generation_gwh'] / bright['ssrd_wm2']

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(bright['temperature_2m_C'], bright['gen_per_irradiance'], alpha=0.05, s=3, c='steelblue', edgecolor='none')
ax.set_xlabel('Temperature (C)')
ax.set_ylabel('PV generation / SSRD (GWh per W/m2)')
ax.set_title('Generation efficiency vs temperature (daytime, ssrd > 100 W/m2)')

# Add binned mean
bins = pd.cut(bright['temperature_2m_C'], bins=20)
binned_mean = bright.groupby(bins, observed=True)['gen_per_irradiance'].mean()
bin_centers = [interval.mid for interval in binned_mean.index]
ax.plot(bin_centers, binned_mean.values, 'r-o', markersize=4, linewidth=2, label='binned mean')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Key observations & notes for preprocessing

Observations will be filled in after running the notebook.

In [ ]:
# Cleanup temporary columns
df.drop(columns=['season', 'hour'], inplace=True)